# Expert review quality — visual summary

Charts from **`rating_sheet_expert1.csv`** (Expert 1 binary rubric).

**Metrics:** prompt adherence, accuracy (acceptable), per-criterion pass rates, by domain.

```bash
pip install pandas matplotlib
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

CSV_PATH = Path("rating_sheet_expert1.csv")
if not CSV_PATH.is_file():
    CSV_PATH = Path("data/reviews/rating_sheet_expert1.csv")

CRITERIA = [
    "topic_relevance",
    "semantic_correctness",
    "answer_key_correctness",
    "question_clarity",
]
CRITERIA_LABELS = {
    "topic_relevance": "Prompt adherence",
    "semantic_correctness": "Semantic correctness",
    "answer_key_correctness": "Answer key",
    "question_clarity": "Clarity",
}

df = pd.read_csv(CSV_PATH)
for col in CRITERIA + ["acceptable"]:
    df[col] = df[col].astype(int)

n = len(df)
acceptable_n = int(df["acceptable"].sum())
accuracy_pct = 100.0 * acceptable_n / n

print(f"File: {CSV_PATH.resolve()}")
print(f"Questions reviewed: {n}")
print(f"Acceptable: {acceptable_n}/{n} ({accuracy_pct:.2f}%)")
print(f"Prompt adherence: {100 * df['topic_relevance'].mean():.2f}%")
df[["item_id", "domain", "acceptable"]].head()

In [ ]:
# 1) Overall criterion pass rates (%)
overall = {CRITERIA_LABELS[c]: 100 * df[c].mean() for c in CRITERIA}
overall["Accuracy\n(acceptable)"] = accuracy_pct

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(
    overall.keys(),
    overall.values(),
    color=["#2ecc71", "#3498db", "#9b59b6", "#1abc9c", "#e67e22"],
)
ax.set_ylim(0, 105)
ax.set_ylabel("Pass rate (%)")
ax.set_title(f"Expert 1 quality metrics (n={n})")
ax.axhline(100, color="gray", ls="--", lw=0.8, alpha=0.5)
for b, v in zip(bars, overall.values()):
    ax.text(b.get_x() + b.get_width() / 2, v + 1, f"{v:.1f}%", ha="center", fontsize=9)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 2) Accuracy & prompt adherence by domain
by_domain = (
    df.groupby("domain")
    .agg(
        n=("item_id", "count"),
        accuracy_pct=("acceptable", lambda s: 100 * s.mean()),
        prompt_adherence_pct=("topic_relevance", lambda s: 100 * s.mean()),
    )
    .reset_index()
    .sort_values("accuracy_pct")
)

x = range(len(by_domain))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([i - w / 2 for i in x], by_domain["prompt_adherence_pct"], width=w, label="Prompt adherence", color="#3498db")
ax.bar([i + w / 2 for i in x], by_domain["accuracy_pct"], width=w, label="Accuracy (acceptable)", color="#e67e22")
ax.set_xticks(list(x))
ax.set_xticklabels(by_domain["domain"], rotation=20, ha="right")
ax.set_ylim(0, 105)
ax.set_ylabel("Pass rate (%)")
ax.set_title("Quality by domain (Expert 1)")
for idx, row in enumerate(by_domain.itertuples()):
    ax.text(idx - w / 2, row.prompt_adherence_pct + 1, f"n={int(row.n)}", ha="center", fontsize=8)
ax.legend()
plt.tight_layout()
plt.savefig("poster_expert_quality_by_domain.png", dpi=200, bbox_inches="tight")
plt.show()
by_domain

In [ ]:
# 3) Acceptable vs not + questions per domain
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

counts = df["acceptable"].value_counts().sort_index()
axes[0].pie(
    counts,
    labels=["Not acceptable", "Acceptable"],
    autopct="%1.1f%%",
    colors=["#e74c3c", "#2ecc71"],
    startangle=90,
)
axes[0].set_title(f"Overall acceptability ({acceptable_n}/{n})")

domain_counts = df["domain"].value_counts().sort_index()
axes[1].barh(domain_counts.index, domain_counts.values, color="#34495e")
axes[1].set_xlabel("Number of questions")
axes[1].set_title("Questions per domain")
plt.tight_layout()
plt.show()

In [ ]:
# 4) Heatmap — criterion pass rate by domain (poster / detail view)
heat = df.groupby("domain")[CRITERIA].mean() * 100
heat.columns = [CRITERIA_LABELS[c] for c in CRITERIA]
heat = heat.sort_index()

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(heat.values, aspect="auto", cmap="YlGn", vmin=80, vmax=100)
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=20, ha="right")
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index)
ax.set_title(f"Criterion pass rate (%) by domain (n={n} MCQs)", fontsize=11)
for i in range(len(heat.index)):
    for j in range(len(heat.columns)):
        ax.text(j, i, f"{heat.values[i, j]:.0f}", ha="center", va="center", fontsize=10)
plt.colorbar(im, ax=ax, label="% pass")
plt.tight_layout()
plt.savefig("poster_expert_heatmap_by_domain.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved: poster_expert_heatmap_by_domain.png")
heat

In [ ]:
# 5) Failed items (acceptable = 0)
failed = df[df["acceptable"] == 0][
    [
        "item_id",
        "domain",
        "subtopic",
        "topic_relevance",
        "semantic_correctness",
        "answer_key_correctness",
        "question_clarity",
        "notes",
    ]
]
if failed.empty:
    print("No failed items.")
else:
    display(failed)

### Poster text — Figure 2

**Title:** Expert evaluation of LLM-generated MCQ quality by domain

**Heatmap:** Four criteria × five domains (**n = 100**, 20/domain). All domains **100%** prompt adherence. Overall accuracy **96%** (96/100).

**Grouped bar chart:** Prompt adherence vs accuracy by domain (saved as `poster_expert_quality_by_domain.png`).